In [ ]:
# ---- Importing Media ---- #
from media import medium_helper

SiN = medium_helper.get_medium(
    "si3n4",
    (0.600, 1.200),
)

SiO2 = medium_helper.get_medium(
    "sio2",
    (0.600, 1.200)
)

Si = medium_helper.get_medium(
    "si",
    (0.600, 1.200)
)

Air = medium_helper.get_medium(
    "air",
    (0.600, 1.200)
)

In [ ]:
# ---- Defining the Geometry of the Grarting Coupler ---- #

from geometries import grating_coupler
from utilities import sim_plotter
from gratingCoupler import GratingConfig, GratingCoupler

from pathlib import Path
import numpy as np

example_config = GratingConfig(
    wavelength_um=1.550,    # Central wavelength (μm)
    theta_deg=-12.0,        # Fiber incidence angle (degrees off-vertical)
    mfd=10.4,               # Mode Field Diameter (Standard SMF-28)
    source_x=3.8419,           # Fiber beam center X position
    source_gap=2.0,         # Distance between fiber source and top cladding

    # Substrate & Layer Geometry
    core_height=0.800,      # Core thickness (μm)
    cladding_height=2.0,    # Air/Cladding buffer height above core
    box_height=4.0,         # Buried Oxide (BOX) thickness
    substrate_height=8.0,   # Silicon handle thickness

    # Grating Teeth Dimensions
    etch_depth=0.5136,       # Partial etch depth (μm)
    period=0.8201,           # Grating pitch/period (μm)
    ff=0.5136,                # Fill factor (50% tooth, 50% groove)
    n_teeth=22,             # Total number of grating teeth
    w_grating=0.800,        # Transverse width for 3D runs

    # Source & Domain Settings
    pol_angle=np.pi/2,          # 0.0 = TE polarization
    buffer=3.0              # Domain margin buffer
)

example_grating_coupler_3d: GratingCoupler = grating_coupler.get_active_simulation(
    config=example_config,
    core_medium=SiN,
    box_medium=SiO2,
    substrate_medium=None,  
    cladding_medium=Air,    # Air-clad (or pass SiO2 for oxide clad)
    is_3d=True           
)

example_grating_coupler_2d : GratingCoupler = grating_coupler.get_active_simulation(
    config=example_config,
    core_medium=SiN,
    box_medium=SiO2,
    substrate_medium=None,
    cladding_medium=Air,
    is_3d=False
)

sim_plotter.display_3d(
    device = example_grating_coupler_3d,
    output_path = Path("example_results/grating_coupler/example_3d.png"),
    show = True,
    save = True
)

sim_plotter.display_2d(
    device=example_grating_coupler_2d,
    output_path=Path("example_results/grating_coupler/example_2d.png"),
    show = True,
    save = True
)

In [ ]:
# ---- Grating Coupler Analytics ---- #

# 1) Coupler efficiency
"""
    The path that is passed to these functions is where the simulation results
    are stored locally after being run on the Tidy3D servers. This ensures that
    if the same configuration is requested more than once, only the first request
    is passed to the Tidy3D servers, and the rest are served from the local data store.

    We have noted that Tidy3D servers do carry out a server-side search to ensure that
    no two simulations are run twice. This layer of local-searching reduces
    latency.

    PROBLEM: As of now the simulation saves are indifferent to the media used to
    build the simulation. This is likely to be corrected with an update to the
    Medium handling architecture.
"""
example_grating_coupler_2d.get_efficiency(Path("example_results/grating_coupler/"))
example_grating_coupler_3d.get_efficiency(Path("example_results/grating_coupler/"))

# 2) Visualize fields within the coupler